# LeetCode #114 — Flatten Binary Tree to Linked List: làm phẳng theo preorder, tại chỗ

**Mức độ:** Medium · **Chủ đề:** Linked List, Stack, Tree, Depth-First Search, Binary Tree

Skeleton trên LeetCode:

```python
class Solution:
    def flatten(self, root: Optional[TreeNode]) -> None:
        """
        Do not return anything, modify root in-place instead.
        """
```

Mục tiêu của notebook:

1. Hiểu yêu cầu: "danh sách liên kết" dùng con trỏ `right` làm `next`, `left` luôn `None`, thứ tự **preorder**.
2. Cách 1: lưu preorder ra list rồi nối lại (`O(n)` bộ nhớ).
3. Cách 2: đệ quy **ngược** (phải → trái → gốc) với con trỏ `prev`.
4. Cách 3 (follow-up `O(1)`): ý tưởng giống **Morris** — nối cây con phải vào nút phải nhất của cây con trái.

# 1. Bài toán là gì?

Cho `root` của cây nhị phân, **làm phẳng** nó thành một "danh sách liên kết":

- Dùng chính lớp `TreeNode`: con trỏ `right` trỏ tới nút **kế tiếp**, `left` luôn là `None`.
- Thứ tự các nút giống thứ tự **duyệt preorder** (gốc → trái → phải) của cây ban đầu.

Sửa **tại chỗ**, không trả về gì.

**Constraints (theo đề):**

- Số nút trong khoảng `[0, 2000]`.
- `-100 <= Node.val <= 100`

**Follow up (theo đề):** làm được **tại chỗ** với `O(1)` bộ nhớ phụ không?

---

## Ví dụ

```text
        1
       / \
      2   5            ->   1 -> 2 -> 3 -> 4 -> 5 -> 6   (qua con trỏ right)
     / \   \
    3   4   6

LeetCode in: [1, null, 2, null, 3, null, 4, null, 5, null, 6]

root = []  ->  []
root = [0] ->  [0]
```

# 2. Kiến thức nền cần nắm trước

Để hiểu trọn lời giải bài này, bạn cần nắm chắc các kiến thức dưới đây. Mỗi phần trình bày: khái niệm là gì, hình minh họa, code mẫu **chạy được**, độ phức tạp và lỗi hay gặp. Phần nào đã vững, bạn có thể đọc lướt.

1. Cây nhị phân (binary tree)
2. Đệ quy (recursion)

Cuối mục có phần **2.3 Áp dụng vào bài này** nối các kiến thức trên với lời giải.

## 2.1 Cây nhị phân (binary tree)

### Cây nhị phân là gì?

Cây nhị phân gồm các **nút**; mỗi nút có một giá trị và tối đa **hai con**: con **trái** và con **phải**. Nút trên cùng là **gốc** (root); nút không có con là **lá** (leaf).

```text
          1          ← gốc (độ sâu 0)
        /   \
       2     3       ← độ sâu 1
      / \     \
     4   5     6     ← lá 4, 5, 6 (độ sâu 2)
```

Thuật ngữ: **cha/con**, **anh em** (cùng cha), **cây con** (một nút cùng toàn bộ con cháu), **độ sâu** (số cạnh từ gốc), **chiều cao** (số tầng / đường dài nhất từ gốc tới lá).

In [1]:
class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right

# Dựng cây ở hình trên bằng tay
goc = TreeNode(1,
               TreeNode(2, TreeNode(4), TreeNode(5)),
               TreeNode(3, None, TreeNode(6)))
print("gốc:", goc.val, "| con trái:", goc.left.val, "| con phải:", goc.right.val,
      "| 3 có con trái?", goc.right.left)

gốc: 1 | con trái: 2 | con phải: 3 | 3 có con trái? None


### Biểu diễn kiểu LeetCode: danh sách theo tầng (level-order)

LeetCode ghi cây thành danh sách đọc từng tầng, từ trái sang phải, `None` cho chỗ trống: cây trên là `[1, 2, 3, 4, 5, None, 6]`. Dựng cây từ danh sách này bằng một **hàng đợi**: lần lượt lấy nút cha ra, gán hai giá trị kế tiếp làm con trái/phải.

In [2]:
from collections import deque

def tao_cay(values):
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    q, i = deque([root]), 1
    while q and i < len(values):
        node = q.popleft()
        if i < len(values) and values[i] is not None:
            node.left = TreeNode(values[i]); q.append(node.left)
        i += 1
        if i < len(values) and values[i] is not None:
            node.right = TreeNode(values[i]); q.append(node.right)
        i += 1
    return root

def ve_cay(node, pre="", la_phai=True, goc_cay=True):
    """In cây nằm ngang để dễ nhìn."""
    if node is None:
        return
    ve_cay(node.right, pre + ("" if goc_cay else ("    " if la_phai else "│   ")), True, False)
    print(pre + ("" if goc_cay else ("┌── " if la_phai else "└── ")) + str(node.val))
    ve_cay(node.left, pre + ("" if goc_cay else ("│   " if la_phai else "    ")), False, False)

ve_cay(tao_cay([1, 2, 3, 4, 5, None, 6]))

    ┌── 6
┌── 3
1
│   ┌── 5
└── 2
    └── 4


(Cây in nằm ngang: gốc ở bên trái, con **phải** ở phía **trên**, con **trái** ở phía **dưới**.)

### Ba cách duyệt theo chiều sâu

Với mỗi nút ta làm ba việc: thăm **nút**, duyệt cây con **trái**, duyệt cây con **phải**. Thứ tự của ba việc cho ba kiểu duyệt:

| Kiểu | Thứ tự | Cây trên |
|---|---|---|
| **Preorder** (tiền thứ tự) | nút → trái → phải | `1 2 4 5 3 6` |
| **Inorder** (trung thứ tự) | trái → nút → phải | `4 2 5 1 3 6` |
| **Postorder** (hậu thứ tự) | trái → phải → nút | `4 5 2 6 3 1` |

Mẹo nhớ: "pre/in/post" chỉ **vị trí của nút** so với hai cây con.

In [3]:
def preorder(node, out):
    if node is None:
        return
    out.append(node.val)        # nút
    preorder(node.left, out)    # trái
    preorder(node.right, out)   # phải

def inorder(node, out):
    if node is None:
        return
    inorder(node.left, out)
    out.append(node.val)
    inorder(node.right, out)

def postorder(node, out):
    if node is None:
        return
    postorder(node.left, out)
    postorder(node.right, out)
    out.append(node.val)

cay = tao_cay([1, 2, 3, 4, 5, None, 6])
for ten, f in [("preorder", preorder), ("inorder", inorder), ("postorder", postorder)]:
    out = []; f(cay, out); print(f"{ten:<9}:", out)

preorder : [1, 2, 4, 5, 3, 6]
inorder  : [4, 2, 5, 1, 3, 6]
postorder: [4, 5, 2, 6, 3, 1]


### Duyệt không đệ quy bằng ngăn xếp

Đệ quy thực chất dùng *call stack*; ta có thể tự quản lý ngăn xếp. Preorder: lấy nút ra, ghi lại, đẩy con **phải trước** rồi con **trái** (để trái được lấy ra trước). Inorder: đi trái hết mức, lấy ra, rồi sang phải.

In [4]:
def preorder_lap(root):
    out, st = [], [root] if root else []
    while st:
        node = st.pop()
        out.append(node.val)
        if node.right: st.append(node.right)
        if node.left:  st.append(node.left)
    return out

def inorder_lap(root):
    out, st, cur = [], [], root
    while cur or st:
        while cur:              # đi trái hết mức
            st.append(cur); cur = cur.left
        cur = st.pop()
        out.append(cur.val)
        cur = cur.right
    return out

print("preorder lặp:", preorder_lap(cay), "| inorder lặp:", inorder_lap(cay))

preorder lặp: [1, 2, 4, 5, 3, 6] | inorder lặp: [4, 2, 5, 1, 3, 6]


### Tư duy đệ quy trên cây

Hầu hết bài về cây giải bằng cách hỏi: "nếu đã biết đáp án cho **cây con trái** và **cây con phải**, ghép lại thế nào?". Ví dụ chiều cao, số nút, tổng giá trị:

In [5]:
def chieu_cao(node):
    if node is None:
        return 0                                   # cây rỗng cao 0
    return 1 + max(chieu_cao(node.left), chieu_cao(node.right))

def so_nut(node):
    return 0 if node is None else 1 + so_nut(node.left) + so_nut(node.right)

print("chiều cao:", chieu_cao(cay), "| số nút:", so_nut(cay))

chiều cao: 3 | số nút: 6


### Lỗi hay gặp

- Quên trường hợp `node is None` → `AttributeError: 'NoneType' object has no attribute 'left'`.
- Nhầm "lá" (không có con nào) với "nút thiếu một con".
- Nhầm thứ tự preorder/inorder/postorder.
- Cây lệch (như danh sách) cao tới `n` → đệ quy sâu `n` tầng.

## 2.2 Đệ quy (recursion)

### Đệ quy là gì?

Một hàm **đệ quy** là hàm tự gọi lại chính nó trên một bài toán **nhỏ hơn** cùng dạng. Mỗi hàm đệ quy có hai phần:

1. **Trường hợp cơ sở (base case)**: bài toán đủ nhỏ để trả lời ngay, không gọi tiếp.
2. **Bước đệ quy**: chia nhỏ, gọi lại, rồi dùng kết quả để trả lời bài lớn.

Ví dụ `n! = n × (n−1)!`, với `0! = 1`.

In [6]:
def giai_thua(n):
    if n == 0:                  # trường hợp cơ sở
        return 1
    return n * giai_thua(n - 1) # bước đệ quy: bài nhỏ hơn
print([giai_thua(n) for n in range(7)])

[1, 1, 2, 6, 24, 120, 720]


### Ngăn xếp lời gọi (call stack)

Mỗi lần gọi hàm, máy tạo một **khung** (frame) chứa biến cục bộ và đặt lên **ngăn xếp lời gọi**. Hàm con chạy xong thì khung của nó bị bỏ, kết quả trả về cho khung bên dưới. Ta in thụt lề theo độ sâu để thấy rõ:

In [7]:
def giai_thua_vet(n, sau=0):
    le = "    " * sau
    print(f"{le}gọi giai_thua({n})")
    if n == 0:
        print(f"{le}→ trả 1 (cơ sở)")
        return 1
    kq = n * giai_thua_vet(n - 1, sau + 1)
    print(f"{le}→ trả {n} × ... = {kq}")
    return kq
_ = giai_thua_vet(3)

gọi giai_thua(3)
    gọi giai_thua(2)
        gọi giai_thua(1)
            gọi giai_thua(0)
            → trả 1 (cơ sở)
        → trả 1 × ... = 1
    → trả 2 × ... = 2
→ trả 3 × ... = 6


### Cách viết một hàm đệ quy đúng

1. **Định nghĩa rõ** hàm trả về gì cho đầu vào nào ("`f(node)` = chiều cao cây gốc `node`").
2. **Tin** rằng lời gọi trên bài nhỏ hơn đã trả đúng (giả thiết quy nạp).
3. Ghép kết quả các bài nhỏ thành kết quả bài lớn.
4. Kiểm tra trường hợp cơ sở được chạm tới với **mọi** đầu vào (mỗi lần gọi phải nhỏ đi).

Ví dụ: tổng các chữ số, lũy thừa nhanh.

In [8]:
def tong_chu_so(n):
    if n < 10:
        return n
    return n % 10 + tong_chu_so(n // 10)

def luy_thua(x, k):              # x^k, chia đôi số mũ
    if k == 0:
        return 1
    nua = luy_thua(x, k // 2)
    return nua * nua * (x if k % 2 else 1)

print("tong_chu_so(98765) =", tong_chu_so(98765), "| luy_thua(3, 13) =", luy_thua(3, 13), "=", 3 ** 13)

tong_chu_so(98765) = 35 | luy_thua(3, 13) = 1594323 = 1594323


### Ghi nhớ kết quả (memoization)

Một số hàm đệ quy gọi **lặp lại** cùng một bài con rất nhiều lần, ví dụ Fibonacci `F(n) = F(n−1) + F(n−2)`: `F(5)` gọi `F(3)` hai lần, `F(2)` ba lần… → `O(2^n)`.

**Memoization**: lưu kết quả đã tính vào dict (hoặc dùng `@lru_cache`), lần sau gặp lại thì trả ngay → mỗi bài con tính đúng một lần.

In [9]:
from functools import lru_cache

dem_goi = 0
def fib_cham(n):
    global dem_goi
    dem_goi += 1
    return n if n < 2 else fib_cham(n - 1) + fib_cham(n - 2)

memo = {}
def fib_memo(n):
    if n < 2:
        return n
    if n not in memo:
        memo[n] = fib_memo(n - 1) + fib_memo(n - 2)
    return memo[n]

@lru_cache(maxsize=None)
def fib_cache(n):
    return n if n < 2 else fib_cache(n - 1) + fib_cache(n - 2)

print("fib_cham(25) =", fib_cham(25), "với", dem_goi, "lần gọi")
print("fib_memo(25) =", fib_memo(25), "| bảng nhớ có", len(memo), "mục")
print("fib_cache(90) =", fib_cache(90))

fib_cham(25) = 75025 với 242785 lần gọi
fib_memo(25) = 75025 | bảng nhớ có 24 mục
fib_cache(90) = 2880067194370816120


### Lỗi hay gặp

- **Thiếu / sai trường hợp cơ sở** → đệ quy vô hạn → `RecursionError` (Python mặc định giới hạn ~1000 tầng).
- Bài con **không nhỏ đi** (ví dụ gọi `f(n)` trong `f(n)`).
- Đệ quy quá sâu với dữ liệu lớn (danh sách 10^5 nút) → đổi sang vòng lặp hoặc tăng `sys.setrecursionlimit` (có rủi ro).
- Dùng list/biến dùng chung giữa các lời gọi mà quên "trả lại" trạng thái (xem backtracking).

In [10]:
def thieu_co_so(n):
    return thieu_co_so(n - 1)
try:
    thieu_co_so(5)
except RecursionError as e:
    print("RecursionError:", str(e)[:45], "...")

RecursionError: maximum recursion depth exceeded ...


## 2.3 Áp dụng vào bài này

Làm phẳng cây theo thứ tự **preorder** thành một "danh sách" đi bằng con phải. Cần hiểu rõ thứ tự preorder (nút → trái → phải) và tư duy **đệ quy** trên cây: làm phẳng hai cây con, rồi gắn cây con trái (đã phẳng) vào giữa nút và cây con phải.

# 3. Nền tảng

In [11]:
from typing import List, Optional
from collections import deque
import random, sys
sys.setrecursionlimit(10000)


class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right


def build_tree(values: list) -> Optional[TreeNode]:
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    q = deque([root])
    i = 1
    while q and i < len(values):
        node = q.popleft()
        if i < len(values) and values[i] is not None:
            node.left = TreeNode(values[i]); q.append(node.left)
        i += 1
        if i < len(values) and values[i] is not None:
            node.right = TreeNode(values[i]); q.append(node.right)
        i += 1
    return root


def preorder_of(root):
    return [root.val] + preorder_of(root.left) + preorder_of(root.right) if root else []


def read_flat(root, limit=5000):
    """Đọc 'danh sách' theo right; kiểm tra mọi left đều None."""
    out, node = [], root
    while node and len(out) < limit:
        assert node.left is None, f"nút {node.val} vẫn còn con trái"
        out.append(node.val)
        node = node.right
    return out

EX = [1, 2, 5, 3, 4, None, 6]
print("preorder:", preorder_of(build_tree(EX)))

preorder: [1, 2, 3, 4, 5, 6]


# 4. Cách 1: Lưu preorder rồi nối lại

In [12]:
def flatten_list(root: Optional[TreeNode]) -> None:
    nodes = []
    def pre(node):
        if node:
            nodes.append(node)
            pre(node.left)
            pre(node.right)
    pre(root)
    for a, b in zip(nodes, nodes[1:]):
        a.left = None
        a.right = b
    if nodes:
        nodes[-1].left = nodes[-1].right = None

t = build_tree(EX); flatten_list(t); print(read_flat(t))

[1, 2, 3, 4, 5, 6]


Lưu **nút** (không phải giá trị) rồi nối lại. `O(n)` bộ nhớ phụ.

**Tại sao phải lưu trước?** Nếu nối trong lúc duyệt preorder, ta sẽ ghi đè `node.right` **trước khi** kịp duyệt cây con phải của nó.

# 5. Cách 2: Đệ quy ngược với `prev`

Ý tưởng: xây danh sách **từ cuối lên đầu**. Thứ tự ngược của preorder là "phải → trái → gốc". Duyệt theo thứ tự đó, giữ `prev` = nút vừa xử lý (sẽ là nút **kế tiếp** của nút hiện tại):

```text
visit(node):
    if not node: return
    visit(node.right)
    visit(node.left)
    node.right = prev
    node.left = None
    prev = node
```

Khi tới `node`, cả cây con phải và trái đã được duyệt xong (đã được làm phẳng và gắn sau), nên ghi đè con trỏ của `node` là an toàn.

In [13]:
def flatten_reverse(root: Optional[TreeNode]) -> None:
    prev = None
    def visit(node):
        nonlocal prev
        if not node:
            return
        visit(node.right)
        visit(node.left)
        node.right = prev
        node.left = None
        prev = node
    visit(root)

t = build_tree(EX); flatten_reverse(t); print(read_flat(t))

[1, 2, 3, 4, 5, 6]


# 6. Cách 3: `O(1)` bộ nhớ — gắn cây phải vào cuối cây trái

Với mỗi nút `cur` (đi dọc theo `right`):

1. Nếu `cur.left` tồn tại: tìm **nút phải nhất** của cây con trái (`pre`) — đó là nút preorder **cuối cùng** của cây con trái, đứng ngay trước cây con phải.
2. `pre.right = cur.right` — gắn cây con phải vào sau nó.
3. `cur.right = cur.left`, `cur.left = None` — chuyển cây con trái sang phải.
4. `cur = cur.right`.

```text
    1                1                1
   / \                \                \
  2   5      ->        2       ->       2
 / \   \              / \                \
3   4   6            3   4                3
                          \                \
                           5                4
                            \                \
                             6                5 -> 6
```

In [14]:
def flatten_o1(root: Optional[TreeNode]) -> None:
    cur = root
    while cur:
        if cur.left:
            pre = cur.left
            while pre.right:
                pre = pre.right
            pre.right = cur.right
            cur.right = cur.left
            cur.left = None
        cur = cur.right

t = build_tree(EX); flatten_o1(t); print(read_flat(t))

[1, 2, 3, 4, 5, 6]


# 7. Mô phỏng cách 3

In [15]:
def show(root):
    """In cây dạng (val, trái, phải) ngắn gọn."""
    def s(n):
        if not n:
            return "·"
        if not n.left and not n.right:
            return str(n.val)
        return f"{n.val}({s(n.left)},{s(n.right)})"
    return s(root)

def trace_o1(values):
    root = build_tree(values)
    cur = root
    print("Ban đầu:", show(root))
    while cur:
        if cur.left:
            pre = cur.left
            while pre.right:
                pre = pre.right
            pre.right = cur.right
            cur.right = cur.left
            cur.left = None
            print(f"  cur = {cur.val}: pre = {pre.val} (phải nhất của cây trái) -> {show(root)}")
        else:
            print(f"  cur = {cur.val}: không có con trái, đi tiếp")
        cur = cur.right
    print("Kết quả:", read_flat(root))
    return root

_ = trace_o1(EX)

Ban đầu: 1(2(3,4),5(·,6))
  cur = 1: pre = 4 (phải nhất của cây trái) -> 1(·,2(3,4(·,5(·,6))))
  cur = 2: pre = 3 (phải nhất của cây trái) -> 1(·,2(·,3(·,4(·,5(·,6)))))
  cur = 3: không có con trái, đi tiếp
  cur = 4: không có con trái, đi tiếp
  cur = 5: không có con trái, đi tiếp
  cur = 6: không có con trái, đi tiếp
Kết quả: [1, 2, 3, 4, 5, 6]


Mỗi nút được đi qua bởi `cur` đúng một lần; vòng `while pre.right` đi qua các cạnh phải của cây con trái — mỗi cạnh tổng cộng chỉ được đi qua hằng số lần, nên vẫn `O(n)`.

# 8. Phiên bản đúng format LeetCode

In [16]:
class Solution:
    def flatten(self, root: Optional[TreeNode]) -> None:
        cur = root
        while cur:
            if cur.left:
                pre = cur.left
                while pre.right:
                    pre = pre.right
                pre.right = cur.right
                cur.right = cur.left
                cur.left = None
            cur = cur.right

In [17]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<18} PASS  {actual}")


def random_tree(n):
    if n == 0:
        return None
    k = random.randint(0, n - 1)
    return TreeNode(random.randint(-9, 9), random_tree(k), random_tree(n - 1 - k))


def run(f, root):
    expected = preorder_of(root)
    ids = []
    def collect(n):
        if n:
            ids.append(id(n)); collect(n.left); collect(n.right)
    collect(root)
    assert f(root) is None
    got = read_flat(root)
    node, got_ids = root, []
    while node:
        got_ids.append(id(node)); node = node.right
    assert got_ids == ids, "phải dùng lại đúng các nút, theo preorder"
    return got, expected


sol = Solution()
got, exp = run(sol.flatten, build_tree(EX)); check("Ví dụ 1", got, [1, 2, 3, 4, 5, 6])
got, exp = run(sol.flatten, build_tree([])); check("Ví dụ 2", got, [])
got, exp = run(sol.flatten, build_tree([0])); check("Ví dụ 3", got, [0])
got, exp = run(sol.flatten, build_tree([1, 2, None, 3])); check("Lệch trái", got, [1, 2, 3])

random.seed(114)
for _ in range(3000):
    n = random.randint(0, 15)
    seed = random.random()
    for f in (sol.flatten, flatten_list, flatten_reverse):
        random.seed(seed); root = random_tree(n)
        got, exp = run(f, root)
        assert got == exp
print("3000 cây ngẫu nhiên: 3 cách đều cho đúng preorder, left = None, dùng lại các nút")

Ví dụ 1            PASS  [1, 2, 3, 4, 5, 6]


Ví dụ 2            PASS  []
Ví dụ 3            PASS  [0]
Lệch trái          PASS  [1, 2, 3]


3000 cây ngẫu nhiên: 3 cách đều cho đúng preorder, left = None, dùng lại các nút


# 9. Vì sao đúng? Độ phức tạp?

**Cách 3 — bất biến:** các nút từ `root` tới `cur` (theo `right`) đã nằm đúng thứ tự preorder, và phần còn lại (cây gốc `cur`) có preorder bằng phần còn lại của preorder ban đầu.

Ở `cur` có con trái: preorder của cây gốc `cur` là `cur, preorder(trái), preorder(phải)`. Nút cuối cùng của `preorder(trái)` là nút **phải nhất** của cây trái (đi phải tới khi hết). Gắn cây phải vào sau nó và chuyển cây trái sang phải → cây mới gốc `cur` có cùng preorder, nhưng `cur` không còn con trái.

```text
             Time    Bộ nhớ phụ
Lưu list     O(n)    O(n)
Đệ quy ngược O(n)    O(h)
Cách 3       O(n)    O(1)
```

# 10. Những lỗi dễ mắc

## Lỗi 1: Nối ngay trong lúc duyệt preorder

In [18]:
def wrong_link_while_preorder(root: Optional[TreeNode]) -> None:
    prev = None
    def pre(node):
        nonlocal prev
        if not node:
            return
        if prev:
            prev.right = node                      # SAI: ghi đè right của prev trước khi duyệt nó
            prev.left = None
        prev = node
        pre(node.left)
        pre(node.right)
    pre(root)

t = build_tree(EX)
try:
    wrong_link_while_preorder(t)
    print("Sai :", read_flat(t, limit=20))
except (AssertionError, RecursionError) as e:
    print("Sai : lỗi", type(e).__name__, str(e)[:60])

Sai : lỗi RecursionError maximum recursion depth exceeded


Khi tới nút `2`, ta đặt `1.right = 2` — **mất** con trỏ tới `5` (cây con phải của `1`). Tệ hơn: khi tới `3`, ta đặt `2.right = 3`; lúc `pre(2)` quay lại để duyệt `2.right`, nó đọc **giá trị mới** (`3`) và vào `3` lần nữa — giờ `prev` là chính `3`, nên `3.right = 3` tạo vòng tự trỏ và đệ quy không bao giờ dừng.

Muốn duyệt preorder và nối cùng lúc, phải **lưu `node.right`** vào biến tạm trước khi đệ quy (Bài 3), hoặc dùng stack.

## Lỗi 2: Quên đặt `left = None`

In [19]:
def wrong_keep_left(root: Optional[TreeNode]) -> None:
    cur = root
    while cur:
        if cur.left:
            pre = cur.left
            while pre.right:
                pre = pre.right
            pre.right = cur.right
            cur.right = cur.left                   # SAI: thiếu cur.left = None
        cur = cur.right

t = build_tree(EX); wrong_keep_left(t)
try:
    read_flat(t)
except AssertionError as e:
    print("Sai :", e)

Sai : nút 1 vẫn còn con trái


## Lỗi 3: Đệ quy ngược nhưng thứ tự trái → phải

Phải duyệt **phải trước trái** (ngược preorder). Thứ tự trái → phải → gốc là postorder, sẽ cho danh sách theo thứ tự sai.

In [20]:
def wrong_reverse_order(root: Optional[TreeNode]) -> None:
    prev = None
    def visit(node):
        nonlocal prev
        if not node:
            return
        visit(node.left)                           # SAI: phải visit(right) trước
        visit(node.right)
        node.right = prev
        node.left = None
        prev = node
    visit(root)

t = build_tree(EX); wrong_reverse_order(t); print("Sai :", read_flat(t), "| Đúng:", [1, 2, 3, 4, 5, 6])

Sai : [1, 5, 6, 2, 4, 3] | Đúng: [1, 2, 3, 4, 5, 6]


## Lỗi 4: Trả về cây mới

Đề yêu cầu sửa **tại chỗ**; trả về giá trị bị LeetCode bỏ qua, và cây gốc không đổi → sai.

# 11. Mẹo nhận diện pattern

- **Làm phẳng / nối lại cây theo một thứ tự duyệt** → hoặc lưu thứ tự rồi nối, hoặc duyệt **ngược** thứ tự đó với `prev`.
- **`O(1)` bộ nhớ trên cây** → nghĩ tới việc "tạm mượn" con trỏ phải của nút phải nhất (Morris, #99, #94).

Bài liên quan: **#426 Convert BST to Sorted Doubly Linked List**, **#897 Increasing Order Search Tree** (làm phẳng theo inorder), **#430 Flatten a Multilevel Doubly Linked List**, **#116 Populating Next Right Pointers**.

# 12. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Áp dụng cách 3 cho cây `[1, 2, 3, None, 4, 5]`. Ghi lại cây sau mỗi lần xử lý một nút có con trái.

## Bài 2 — Điền chỗ trống (cách 3)

In [21]:
def exercise_2_flatten(root: Optional[TreeNode]) -> None:
    cur = root
    while cur:
        if cur.left:
            # TODO: tìm pre; gắn cur.right vào pre.right; chuyển trái sang phải; left = None
            pass
        cur = cur.right


# t = build_tree([1, 2, 5, 3, 4, None, 6]); exercise_2_flatten(t); assert read_flat(t) == [1, 2, 3, 4, 5, 6]
# t = build_tree([]); exercise_2_flatten(t); assert read_flat(t) == []

## Bài 3 — Biến thể: preorder + nối bằng stack

Làm phẳng bằng stack: đẩy gốc; mỗi lần lấy `node`, đẩy `node.right` rồi `node.left`; nối `prev.right = node`.

In [22]:
def exercise_3_flatten_stack(root: Optional[TreeNode]) -> None:
    # TODO
    pass


# t = build_tree([1, 2, 5, 3, 4, None, 6]); exercise_3_flatten_stack(t); assert read_flat(t) == [1, 2, 3, 4, 5, 6]
# t = build_tree([1, 2, None, 3]); exercise_3_flatten_stack(t); assert read_flat(t) == [1, 2, 3]

## Bài 4 — Biến thể: Increasing Order Search Tree (LeetCode #897)

Làm phẳng một **BST** theo **inorder** (tăng dần): mỗi nút chỉ có con phải. Trả về gốc mới (nút nhỏ nhất).

```python
increasing_bst([5, 3, 6, 2, 4, None, 8, 1, None, None, None, 7, 9]) -> 1 -> 2 -> ... -> 9
```

In [23]:
def exercise_4_increasing_bst(root: Optional[TreeNode]) -> Optional[TreeNode]:
    # TODO
    pass


# assert read_flat(exercise_4_increasing_bst(build_tree([5, 3, 6, 2, 4, None, 8, 1, None, None, None, 7, 9]))) == [1, 2, 3, 4, 5, 6, 7, 8, 9]
# assert read_flat(exercise_4_increasing_bst(build_tree([5, 1, 7]))) == [1, 5, 7]

## Bài 5 — Mini challenge: khôi phục cây từ danh sách đã làm phẳng?

Sau khi làm phẳng, chỉ còn thứ tự preorder. Chứng minh bằng ví dụ rằng **không thể** khôi phục cây gốc từ một mình preorder — viết hàm đếm số cây nhị phân **khác cấu trúc** có cùng preorder `1, 2, …, n`.

Gợi ý nhanh: với preorder cố định, mỗi cấu trúc cây cho một preorder khác nhau? Thử với `n = 3`. Số cấu trúc chính là số Catalan.

In [24]:
def exercise_5_count_trees_same_preorder(n: int) -> int:
    # TODO
    pass


# assert exercise_5_count_trees_same_preorder(3) == 5
# assert [exercise_5_count_trees_same_preorder(k) for k in range(1, 7)] == [1, 2, 5, 14, 42, 132]

# 13. Gợi ý

**Bài 1**
- Mức 1: chạy `trace_o1([1, 2, 3, None, 4, 5])` sau khi tự làm.

**Bài 3**
- Mức 1: `prev = None`; `while stack: node = stack.pop()`; nếu `prev`: `prev.right = node; prev.left = None`.
- Mức 2: đẩy con **phải** trước con **trái** (để trái ra trước) — trước khi sửa con trỏ của `node`.

**Bài 4**
- Mức 1: dummy + `tail`; duyệt inorder: `node.left = None; tail.right = node; tail = node`.

**Bài 5**
- Mức 1: preorder cố định là `1..n`; mọi cấu trúc cây `n` nút gán giá trị theo preorder đều cho đúng preorder đó.
- Mức 2: vậy số cây = số cấu trúc cây nhị phân `n` nút = Catalan `C_n`. Sinh thử để kiểm chứng.

# 14. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này.

---

## Đáp án Bài 1

In [25]:
_ = trace_o1([1, 2, 3, None, 4, 5])

Ban đầu: 1(2(·,4),3(5,·))
  cur = 1: pre = 4 (phải nhất của cây trái) -> 1(·,2(·,4(·,3(5,·))))
  cur = 2: không có con trái, đi tiếp
  cur = 4: không có con trái, đi tiếp
  cur = 3: pre = 5 (phải nhất của cây trái) -> 1(·,2(·,4(·,3(·,5))))
  cur = 5: không có con trái, đi tiếp
Kết quả: [1, 2, 4, 3, 5]


## Đáp án Bài 2

In [26]:
def answer_2_flatten(root: Optional[TreeNode]) -> None:
    flatten_o1(root)

t = build_tree([1, 2, 5, 3, 4, None, 6]); answer_2_flatten(t); assert read_flat(t) == [1, 2, 3, 4, 5, 6]
t = build_tree([]); answer_2_flatten(t); assert read_flat(t) == []
print("Bài 2 PASS (giống flatten_o1 ở mục 6)")

Bài 2 PASS (giống flatten_o1 ở mục 6)


## Đáp án Bài 3

In [27]:
def answer_3_flatten_stack(root: Optional[TreeNode]) -> None:
    if not root:
        return
    stack, prev = [root], None
    while stack:
        node = stack.pop()
        if node.right:
            stack.append(node.right)
        if node.left:
            stack.append(node.left)
        if prev:
            prev.right = node
            prev.left = None
        prev = node
    prev.left = prev.right = None

t = build_tree([1, 2, 5, 3, 4, None, 6]); answer_3_flatten_stack(t); assert read_flat(t) == [1, 2, 3, 4, 5, 6]
t = build_tree([1, 2, None, 3]); answer_3_flatten_stack(t); assert read_flat(t) == [1, 2, 3]
for _ in range(1000):
    root = random_tree(random.randint(1, 12))
    exp = preorder_of(root)
    answer_3_flatten_stack(root)
    assert read_flat(root) == exp
print("Bài 3 PASS")

Bài 3 PASS


Đẩy hai con vào stack **trước khi** sửa `prev.right` — vì `prev.right = node` có thể ghi đè con phải của `prev`, nhưng con phải đó đã nằm an toàn trong stack.

## Đáp án Bài 4

In [28]:
def answer_4_increasing_bst(root: Optional[TreeNode]) -> Optional[TreeNode]:
    dummy = tail = TreeNode()
    def inorder(node):
        nonlocal tail
        if not node:
            return
        inorder(node.left)
        node.left = None
        tail.right = node
        tail = node
        inorder(node.right)
    inorder(root)
    return dummy.right

assert read_flat(answer_4_increasing_bst(build_tree([5, 3, 6, 2, 4, None, 8, 1, None, None, None, 7, 9]))) == [1, 2, 3, 4, 5, 6, 7, 8, 9]
assert read_flat(answer_4_increasing_bst(build_tree([5, 1, 7]))) == [1, 5, 7]
print("Bài 4 PASS")

Bài 4 PASS


Lệnh `tail.right = node` ghi đè con phải của nút **trước** `node` theo inorder. An toàn vì con phải của mỗi nút đã được đọc (khi gọi `inorder(node.right)`) **trước** khi nút kế tiếp của nó ghi đè con trỏ đó.

## Đáp án Bài 5

In [29]:
def answer_5_count_trees_same_preorder(n: int) -> int:
    def shapes(k):
        if k == 0:
            return [None]
        out = []
        for left in range(k):
            for L in shapes(left):
                for R in shapes(k - 1 - left):
                    out.append((L, R))
        return out
    trees = shapes(n)
    def assign(shape, it):
        if shape is None:
            return None
        node = TreeNode(next(it))
        node.left = assign(shape[0], it)
        node.right = assign(shape[1], it)
        return node
    preorders = {tuple(preorder_of(assign(s, iter(range(1, n + 1))))) for s in trees}
    assert preorders == {tuple(range(1, n + 1))}          # mọi cấu trúc đều cho cùng preorder
    return len(trees)

assert answer_5_count_trees_same_preorder(3) == 5
assert [answer_5_count_trees_same_preorder(k) for k in range(1, 7)] == [1, 2, 5, 14, 42, 132]
print("Bài 5 PASS: preorder một mình không đủ để khôi phục cây (cần thêm inorder như #105)")

Bài 5 PASS: preorder một mình không đủ để khôi phục cây (cần thêm inorder như #105)


# 15. Tổng kết

> Làm phẳng theo preorder. Cách gọn nhất `O(1)`: với mỗi `cur` có con trái, tìm nút phải nhất `pre` của cây trái, gắn `pre.right = cur.right`, rồi `cur.right = cur.left; cur.left = None`, đi tiếp theo `right`. Hoặc đệ quy ngược (phải → trái → gốc) với `prev`.

---

## Checklist trước khi nộp

- [ ] Mọi `left` đều `None`?
- [ ] Không ghi đè `right` trước khi duyệt cây con phải?
- [ ] Đệ quy ngược: **phải trước trái**?
- [ ] Sửa tại chỗ, không trả về?